# CatBoost Model Experiment: 후보 CONTENT_ID 랭킹

Issue #7 v1 운영형 대안 실험이다. 기존 여행자 입력/여행 조건 입력 스키마는 유지하고, `CONTENT_ID` 후보별 feature를 만들어 CatBoost가 다음 방문지 후보를 점수화한다.

- 입력: `data/processed/total_input.csv`, `data/processed/total_travel_seq_with_contentid.csv`
- 학습 방식: 실제 다음 방문지 1개 + negative sampled 후보 N개로 binary relevance 학습
- 출력: 사용자/조건/context에 대한 추천 후보 `CONTENT_ID` top-k
- 서버 장점: `torch` 없이 `catboost` CPU 추론만 사용

In [5]:
!nvidia-smi

Sat Sep  5 11:53:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P0             30W /   70W |     199MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!pip install catboost

## 1. Import & Settings

In [7]:
from __future__ import annotations

import json
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
ARTIFACT_DIR = ROOT / "artifacts" / "catboost_model_experiment"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "total_input.csv"
SEQ_PATH = DATA_DIR / "total_travel_seq_with_contentid.csv"

SEED = 42
MIN_SEQUENCE_LEN = 2
MAX_PREFIX_LEN = 20
NEGATIVES_PER_POSITIVE = 12
CANDIDATE_POOL_SIZE = 120
TOP_KS = (5, 10)

random.seed(SEED)
np.random.seed(SEED)

## 2. Load Data & Build CONTENT_ID Sequences

In [8]:
input_df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
seq_df = pd.read_csv(SEQ_PATH, encoding="utf-8-sig")

def normalize_content_id(series: pd.Series) -> pd.Series:
    return series.astype("string").str.replace(r"\.0$", "", regex=True)


matched_seq = seq_df.loc[seq_df["CONTENT_ID"].notna()].copy()
matched_seq["CONTENT_ID"] = normalize_content_id(matched_seq["CONTENT_ID"])
matched_seq["CONTENT_TYPE_ID"] = matched_seq["CONTENT_TYPE_ID"].astype("string").str.replace(r"\.0$", "", regex=True)
matched_seq = matched_seq.sort_values(["travel_id", "day_index", "visit_order"], kind="mergesort")

trip_sequences = (
    matched_seq.groupby("travel_id")["CONTENT_ID"]
    .apply(lambda values: [str(value) for value in values if pd.notna(value)])
    .reset_index(name="content_sequence")
)
trip_sequences["sequence_len"] = trip_sequences["content_sequence"].map(len)
trip_sequences = trip_sequences.loc[trip_sequences["sequence_len"] >= MIN_SEQUENCE_LEN].copy()

model_input = input_df.merge(
    trip_sequences,
    left_on="trip_id",
    right_on="travel_id",
    how="inner",
    validate="one_to_one",
)

coverage = {
    "input_trips": int(input_df["trip_id"].nunique()),
    "sequence_rows": int(len(seq_df)),
    "content_rows": int(seq_df["CONTENT_ID"].notna().sum()),
    "usable_trips": int(len(model_input)),
}
coverage

{'input_trips': 7730,
 'sequence_rows': 71649,
 'content_rows': 4922,
 'usable_trips': 1337}

In [9]:
content_meta = (
    matched_seq.sort_values(["CONTENT_ID", "TOURAPI_MATCH_SCORE"], ascending=[True, False], na_position="last")
    .groupby("CONTENT_ID", as_index=False)
    .agg(
        content_type_id=("CONTENT_TYPE_ID", "first"),
        candidate_x=("X_COORD", "median"),
        candidate_y=("Y_COORD", "median"),
        candidate_sgg=("SGG_CD", "first"),
        candidate_visit_name=("visit_area_nm", "first"),
    )
)
content_meta_map = content_meta.set_index("CONTENT_ID").to_dict(orient="index")

global_popularity = matched_seq["CONTENT_ID"].value_counts()
global_candidates = global_popularity.index.tolist()

area_candidates = (
    model_input[["area_code", "content_sequence"]]
    .explode("content_sequence")
    .rename(columns={"content_sequence": "CONTENT_ID"})
    .groupby("area_code")["CONTENT_ID"]
    .apply(lambda values: values.value_counts().index.tolist())
    .to_dict()
)

print(f"unique CONTENT_ID: {len(global_candidates):,}")
content_meta.head()

unique CONTENT_ID: 972


,CONTENT_ID,content_type_id,candidate_x,candidate_y,candidate_sgg,candidate_visit_name
0,1013716,38,129.060815,35.154467,NaN,서면 젊음의 거리
1,1018702,12,129.028890,35.099449,NaN,국제시장 먹자골목
2,1019182,12,126.994384,37.533527,NaN,이태원 앤티크 가구거리
3,1021339,12,126.998865,37.499102,NaN,서래 마을 이야기
4,1022243,14,126.920161,37.548521,NaN,롤링홀


## 3. Train / Valid / Test Split

In [10]:
train_trips, temp_trips = train_test_split(model_input["trip_id"], test_size=0.30, random_state=SEED, shuffle=True)
valid_trips, test_trips = train_test_split(temp_trips, test_size=0.50, random_state=SEED, shuffle=True)

split_sets = {"train": set(train_trips), "valid": set(valid_trips), "test": set(test_trips)}
assert split_sets["train"].isdisjoint(split_sets["valid"])
assert split_sets["train"].isdisjoint(split_sets["test"])
assert split_sets["valid"].isdisjoint(split_sets["test"])

train_df = model_input[model_input["trip_id"].isin(split_sets["train"])].reset_index(drop=True)
valid_df = model_input[model_input["trip_id"].isin(split_sets["valid"])].reset_index(drop=True)
test_df = model_input[model_input["trip_id"].isin(split_sets["test"])].reset_index(drop=True)

split_sizes = {"train": len(train_df), "valid": len(valid_df), "test": len(test_df)}

# Candidate metadata and popularity are fitted from train split only to avoid leakage.
train_matched_seq = matched_seq[matched_seq["travel_id"].isin(split_sets["train"])].copy()
content_meta = (
    train_matched_seq.sort_values(["CONTENT_ID", "TOURAPI_MATCH_SCORE"], ascending=[True, False], na_position="last")
    .groupby("CONTENT_ID", as_index=False)
    .agg(
        content_type_id=("CONTENT_TYPE_ID", "first"),
        candidate_x=("X_COORD", "median"),
        candidate_y=("Y_COORD", "median"),
        candidate_sgg=("SGG_CD", "first"),
        candidate_visit_name=("visit_area_nm", "first"),
    )
)
content_meta_map = content_meta.set_index("CONTENT_ID").to_dict(orient="index")
global_popularity = train_matched_seq["CONTENT_ID"].value_counts()
global_candidates = global_popularity.index.tolist()
area_candidates = (
    train_df[["area_code", "content_sequence"]]
    .explode("content_sequence")
    .rename(columns={"content_sequence": "CONTENT_ID"})
    .groupby("area_code")["CONTENT_ID"]
    .apply(lambda values: values.value_counts().index.tolist())
    .to_dict()
)

{"split_sizes": split_sizes, "train_candidate_count": len(global_candidates)}

{'split_sizes': {'train': 935, 'valid': 201, 'test': 201},
 'train_candidate_count': 760}

## 4. Candidate Ranking Dataset

In [11]:
BASE_INPUT_COLUMNS = [column for column in input_df.columns if column != "trip_id"]

def prefix_stats(prefix: list[str]) -> dict:
    prefix = prefix[-MAX_PREFIX_LEN:]
    last_content_id = prefix[-1] if prefix else "missing"
    last_meta = content_meta_map.get(last_content_id, {})
    unique_prefix = set(prefix)
    return {
        "prefix_len": len(prefix),
        "prefix_unique_count": len(unique_prefix),
        "last_content_id": last_content_id,
        "last_content_type_id": str(last_meta.get("content_type_id", "missing")),
        "last_x": last_meta.get("candidate_x", np.nan),
        "last_y": last_meta.get("candidate_y", np.nan),
    }


def candidate_features(base_row: pd.Series, prefix: list[str], candidate_id: str, label: int | None = None) -> dict:
    meta = content_meta_map.get(candidate_id, {})
    stats = prefix_stats(prefix)
    last_x = stats["last_x"]
    last_y = stats["last_y"]
    candidate_x = meta.get("candidate_x", np.nan)
    candidate_y = meta.get("candidate_y", np.nan)
    if pd.notna(last_x) and pd.notna(last_y) and pd.notna(candidate_x) and pd.notna(candidate_y):
        approx_distance = float(np.sqrt((float(candidate_x) - float(last_x)) ** 2 + (float(candidate_y) - float(last_y)) ** 2))
    else:
        approx_distance = np.nan

    row = {column: base_row.get(column, np.nan) for column in BASE_INPUT_COLUMNS}
    row.update(stats)
    row.update(
        {
            "candidate_content_id": str(candidate_id),
            "candidate_content_type_id": str(meta.get("content_type_id", "missing")),
            "candidate_sgg": str(meta.get("candidate_sgg", "missing")),
            "candidate_visit_name": str(meta.get("candidate_visit_name", "missing")),
            "candidate_global_count": int(global_popularity.get(candidate_id, 0)),
            "candidate_seen_in_prefix": int(candidate_id in set(prefix)),
            "candidate_same_as_last": int(candidate_id == stats["last_content_id"]),
            "last_to_candidate_distance": approx_distance,
        }
    )
    if label is not None:
        row["label"] = int(label)
    return row


def candidate_pool_for_row(base_row: pd.Series, exclude: set[str], pool_size: int = CANDIDATE_POOL_SIZE) -> list[str]:
    area_pool = area_candidates.get(base_row.get("area_code"), [])
    merged = []
    for candidate_id in [*area_pool, *global_candidates]:
        if candidate_id not in exclude and candidate_id not in merged:
            merged.append(candidate_id)
        if len(merged) >= pool_size:
            break
    return merged


def build_training_rows(df: pd.DataFrame, negatives_per_positive: int = NEGATIVES_PER_POSITIVE) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    rows = []
    for _, base_row in df.iterrows():
        sequence = [str(value) for value in base_row["content_sequence"]]
        for target_pos in range(1, len(sequence)):
            prefix = sequence[max(0, target_pos - MAX_PREFIX_LEN):target_pos]
            positive_id = sequence[target_pos]
            rows.append(candidate_features(base_row, prefix, positive_id, label=1))

            exclude = set(prefix) | {positive_id}
            pool = candidate_pool_for_row(base_row, exclude, pool_size=max(CANDIDATE_POOL_SIZE, negatives_per_positive * 4))
            if len(pool) > negatives_per_positive:
                pool = rng.choice(pool, size=negatives_per_positive, replace=False).tolist()
            for negative_id in pool:
                rows.append(candidate_features(base_row, prefix, str(negative_id), label=0))
    return pd.DataFrame(rows)


train_rank_df = build_training_rows(train_df)
valid_rank_df = build_training_rows(valid_df)
test_rank_df = build_training_rows(test_df)

{
    "train_rows": len(train_rank_df),
    "valid_rows": len(valid_rank_df),
    "test_rows": len(test_rank_df),
    "positive_rate": float(train_rank_df["label"].mean()),
}

{'train_rows': 27209,
 'valid_rows': 5382,
 'test_rows': 5395,
 'positive_rate': 0.07692307692307693}

In [12]:
FEATURE_COLUMNS = [column for column in train_rank_df.columns if column != "label"]
CAT_FEATURES = [
    column
    for column in FEATURE_COLUMNS
    if train_rank_df[column].dtype == "object" or str(train_rank_df[column].dtype).startswith("string")
]

for frame in (train_rank_df, valid_rank_df, test_rank_df):
    frame[CAT_FEATURES] = frame[CAT_FEATURES].fillna("missing").astype(str)

CAT_FEATURES

['area_code',
 'p0_gender',
 'p0_style',
 'p0_preferred',
 'last_content_id',
 'last_content_type_id',
 'candidate_content_id',
 'candidate_content_type_id',
 'candidate_sgg',
 'candidate_visit_name']

## 5. Train CatBoost Candidate Scorer

In [13]:
train_pool = Pool(train_rank_df[FEATURE_COLUMNS], label=train_rank_df["label"], cat_features=CAT_FEATURES)
valid_pool = Pool(valid_rank_df[FEATURE_COLUMNS], label=valid_rank_df["label"], cat_features=CAT_FEATURES)

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=800,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=8,
    random_seed=SEED,
    auto_class_weights="Balanced",
    od_type="Iter",
    od_wait=50,
    verbose=100,
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)


0:	test: 0.8566837	best: 0.8566837 (0)	total: 120ms	remaining: 1m 36s
100:	test: 0.9055837	best: 0.9055837 (100)	total: 7.99s	remaining: 55.3s
200:	test: 0.9230282	best: 0.9232008 (199)	total: 14.6s	remaining: 43.5s
300:	test: 0.9314541	best: 0.9314541 (300)	total: 23.1s	remaining: 38.3s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.932848005
bestIteration = 338

Shrink model to first 339 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=6, eval_metric='AUC', iterations=800, l2_leaf_reg=8, learning_rate=0.05, loss_function='Logloss', od_type='Iter', od_wait=50, random_seed=42, verbose=100)

## 6. Top-K Recommendation & Metrics

In [14]:
def recommend_content_ids(
    base_row: pd.Series,
    prefix: list[str] | None = None,
    top_k: int = 10,
    scoring_model: CatBoostClassifier | None = None,
) -> pd.DataFrame:
    scoring_model = model if scoring_model is None else scoring_model
    prefix = [] if prefix is None else [str(value) for value in prefix]
    exclude = set(prefix)
    pool = candidate_pool_for_row(base_row, exclude, pool_size=CANDIDATE_POOL_SIZE)
    candidate_df = pd.DataFrame([candidate_features(base_row, prefix, candidate_id) for candidate_id in pool])
    candidate_df[CAT_FEATURES] = candidate_df[CAT_FEATURES].fillna("missing").astype(str)
    scores = scoring_model.predict_proba(candidate_df[FEATURE_COLUMNS])[:, 1]
    result = candidate_df[["candidate_content_id", "candidate_content_type_id", "candidate_sgg", "candidate_visit_name"]].copy() if "candidate_visit_name" in candidate_df.columns else candidate_df[["candidate_content_id", "candidate_content_type_id", "candidate_sgg"]].copy()
    result["score"] = scores
    return result.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)


def evaluate_next_place_topk(df: pd.DataFrame, ks: tuple[int, ...] = TOP_KS) -> dict[str, float]:
    recalls = {k: [] for k in ks}
    mrrs = {k: [] for k in ks}
    ndcgs = {k: [] for k in ks}
    for _, base_row in df.iterrows():
        sequence = [str(value) for value in base_row["content_sequence"]]
        for target_pos in range(1, len(sequence)):
            prefix = sequence[max(0, target_pos - MAX_PREFIX_LEN):target_pos]
            target = sequence[target_pos]
            ranked = recommend_content_ids(base_row, prefix=prefix, top_k=max(ks))["candidate_content_id"].tolist()
            for k in ks:
                topk = ranked[:k]
                hit = target in topk
                recalls[k].append(float(hit))
                if hit:
                    rank = topk.index(target) + 1
                    mrrs[k].append(1.0 / rank)
                    ndcgs[k].append(1.0 / np.log2(rank + 1))
                else:
                    mrrs[k].append(0.0)
                    ndcgs[k].append(0.0)
    metrics = {}
    for k in ks:
        metrics[f"recall@{k}"] = float(np.mean(recalls[k]))
        metrics[f"mrr@{k}"] = float(np.mean(mrrs[k]))
        metrics[f"ndcg@{k}"] = float(np.mean(ndcgs[k]))
    return metrics


valid_metrics = evaluate_next_place_topk(valid_df)
test_metrics = evaluate_next_place_topk(test_df)
{"valid": valid_metrics, "test": test_metrics}

{'valid': {'recall@5': 0.33816425120772947,
  'mrr@5': 0.2067230273752013,
  'ndcg@5': 0.23935106154048036,
  'recall@10': 0.463768115942029,
  'mrr@10': 0.2232085346215781,
  'ndcg@10': 0.2796870216992364},
 'test': {'recall@5': 0.30120481927710846,
  'mrr@5': 0.19654618473895583,
  'ndcg@5': 0.22262178386290749,
  'recall@10': 0.39518072289156625,
  'mrr@10': 0.20864601262191623,
  'ndcg@10': 0.2525479514340749}}

In [18]:
example_user_condition = test_df.iloc[0].copy()
print(example_user_condition)

trip_id                      e_e000662
area_code                      central
trip_days                            2
theme                                1
has_child                            0
has_elderly                          0
has_disabled                         0
companion_count                      0
p0_age                              30
p0_gender                            남
p0_style               6;3;2;3;3;3;5;3
p0_home                             41
p0_preferred         50110;26350;46130
p1_age                             NaN
p1_gender                          NaN
p1_style                           NaN
p1_home                            NaN
p1_preferred                       NaN
travel_id                    e_e000662
content_sequence    [2500229, 3305576]
sequence_len                         2
Name: 0, dtype: object


In [15]:
example_user_condition = test_df.iloc[0].copy()
example_prefix = example_user_condition["content_sequence"][:1]
recommend_content_ids(example_user_condition, prefix=example_prefix, top_k=10)

,candidate_content_id,candidate_content_type_id,candidate_sgg,candidate_visit_name,score
0,264512,12,nan,인천 차이나타운,0.481496
1,2003909,12,1171010200.0,롯데월드타워 롯데월드몰,0.435506
2,1938168,12,nan,송월동 동화마을,0.399759
3,2906334,38,nan,송도센트럴파크,0.377315
4,2780309,39,nan,바다 앞 테라스,0.376280
5,128767,12,nan,을왕리 해수욕장,0.373891
6,129703,14,nan,국립중앙박물관,0.363171
7,2708109,38,nan,더 현대 서울,0.356037
8,2768101,39,nan,동해 막국수,0.353188
9,2848938,39,nan,카페 아키라,0.349396


## 7. Save Artifacts

In [16]:
model.save_model(str(ARTIFACT_DIR / "catboost_contentid_ranker.cbm"))

candidate_metadata = {
    "content_meta": content_meta.to_dict(orient="records"),
    "global_candidates": global_candidates,
    "area_candidates": area_candidates,
}
with open(ARTIFACT_DIR / "candidate_metadata.pkl", "wb") as fp:
    pickle.dump(candidate_metadata, fp)

config = {
    "seed": SEED,
    "min_sequence_len": MIN_SEQUENCE_LEN,
    "max_prefix_len": MAX_PREFIX_LEN,
    "negatives_per_positive": NEGATIVES_PER_POSITIVE,
    "candidate_pool_size": CANDIDATE_POOL_SIZE,
    "feature_columns": FEATURE_COLUMNS,
    "cat_features": CAT_FEATURES,
    "base_input_columns": BASE_INPUT_COLUMNS,
    "input_path": str(INPUT_PATH.relative_to(ROOT)),
    "sequence_path": str(SEQ_PATH.relative_to(ROOT)),
}
with open(ARTIFACT_DIR / "train_config.json", "w", encoding="utf-8") as fp:
    json.dump(config, fp, ensure_ascii=False, indent=2)

metrics = {
    "coverage": coverage,
    "split_sizes": split_sizes,
    "rank_rows": {
        "train": len(train_rank_df),
        "valid": len(valid_rank_df),
        "test": len(test_rank_df),
    },
    "valid": valid_metrics,
    "test": test_metrics,
}
with open(ARTIFACT_DIR / "metrics.json", "w", encoding="utf-8") as fp:
    json.dump(metrics, fp, ensure_ascii=False, indent=2)

sorted(path.name for path in ARTIFACT_DIR.iterdir())

['candidate_metadata.pkl',
 'catboost_contentid_ranker.cbm',
 'metrics.json',
 'train_config.json']

## 8. Reload Check

In [17]:
loaded_model = CatBoostClassifier()
loaded_model.load_model(str(ARTIFACT_DIR / "catboost_contentid_ranker.cbm"))

with open(ARTIFACT_DIR / "candidate_metadata.pkl", "rb") as fp:
    loaded_candidate_metadata = pickle.load(fp)

sample_candidates = recommend_content_ids(example_user_condition, prefix=example_prefix, top_k=5, scoring_model=loaded_model)
sample_candidates

,candidate_content_id,candidate_content_type_id,candidate_sgg,candidate_visit_name,score
0,264512,12,nan,인천 차이나타운,0.481496
1,2003909,12,1171010200.0,롯데월드타워 롯데월드몰,0.435506
2,1938168,12,nan,송월동 동화마을,0.399759
3,2906334,38,nan,송도센트럴파크,0.377315
4,2780309,39,nan,바다 앞 테라스,0.376280
